In [ ]:
# Celda 1
# Paths and parameters:
%cd ..

LOD3_JSON = "data/lod3_meshes.json"
JSON_FILE = "data/lod3_texturas_meshes.json"
BLEND_FILE = "blender/lod3_texturas.blend"
GLB_FILE = "blender/lod3_texturas.glb"
BLENDER = "C:/Program Files/Blender Foundation/Blender 5.2/blender.exe"
BPY_SCRIPT = "scripts/meshes_to_blender.py"
ORIGIN = (440000.0, 4473700.0, 640.0)
RESOLUTION = 0.025   # Meters per pixel of the rectified photos (09_rectificar.ipynb)

TOLEDO_LINE = [(439988.6, 4473740.7), (439984.4, 4473757.7), (439978.7, 4473780.4), (439979.4, 4473792.5),
               (439980.8, 4473819.3), (439982.0, 4473827.4), (439984.2, 4473842.1), (439985.5, 4473850.4)]

# Textured facades: rectified photo (and its window, as in 09_rectificar), the table of elements that get
# the photo, and the parts of the wall covered by the photo (u_from, u_to, z_from, z_to), cut to the
# outline of the building so that no sky goes onto a wall. Licenses: CC BY-SA (academic use, see SOURCES.md).
TEXTURED = [
    {
        "image": "data/fachadas/toledo_instituto_u41-80.jpg",
        "u_left": 80.5, "z_top": 662.5,
        "facade": "data/fachadas/toledo_instituto_u41-80.csv",
        "line": TOLEDO_LINE,
        "walls": [(40.9, 79.9, 644.0, 661.2)],
    },
    {
        "image": "data/fachadas/toledo_colegiata_u80-111.jpg",
        "u_left": 112.0, "z_top": 684.5,
        "facade": "data/fachadas/toledo_colegiata_u80-111.csv",
        "line": TOLEDO_LINE,
        "walls": [(79.9, 111.3, 644.0, 663.6),     # Lower body up to the top of the balustrade
                  (103.0, 111.3, 663.6, 681.5),    # North tower
                  (79.9, 88.6, 663.6, 681.5)],     # South tower
    },
]
WALL_OFFSET = 0.10   # The photo plane is 10 cm in front of the facade line (LOD2 walls stick out up to 6.5 cm; kept elements stick out more than 10 cm)

In [ ]:
# Celda 2
import os
import json
import subprocess
import shapely
from shapely.geometry import LineString
from PIL import Image

In [ ]:
# Celda 3
# Texture coordinate of a point of the facade (u, z) in the rectified photo:
# (u = u_left, z = z_top) is the top left corner of the image -> (0, 1); the bottom right corner -> (1, 0).
def photo_uv(texture, u, z):
    width_px, height_px = texture["size"]
    column = (texture["u_left"] - u) / RESOLUTION
    row = (texture["z_top"] - z) / RESOLUTION
    return (column / width_px, 1 - row / height_px)

for texture in TEXTURED:
    texture["size"] = Image.open(texture["image"]).size
    texture["path"] = os.path.abspath(texture["image"])
    print(texture["image"], texture["size"])

In [ ]:
# Celda 4
# Photo plane over one part of the wall: a strip that follows the facade line (it bends where the line bends).
def wall_plane(texture, u_from, u_to, z_from, z_to):
    line = LineString(texture["line"])
    cuts = [u_from]
    for corner in list(line.coords)[1:-1]:
        u_corner = line.project(shapely.Point(corner))
        if u_from < u_corner < u_to:
            cuts.append(u_corner)
    cuts.append(u_to)
    x0, y0, z0 = ORIGIN
    vertices = []
    uv = []
    for u in cuts:
        # Point of the line at u, moved WALL_OFFSET out of the wall (to the left of the line direction):
        a = line.interpolate(max(u - 0.5, 0))
        b = line.interpolate(min(u + 0.5, line.length))
        length = a.distance(b)
        dx, dy = (b.x - a.x) / length, (b.y - a.y) / length
        p = line.interpolate(u)
        for z in [z_from, z_to]:
            vertices.append((p.x - dy * WALL_OFFSET - x0, p.y + dx * WALL_OFFSET - y0, z - z0))
            uv.append(photo_uv(texture, u, z))
    faces = []
    for k in range(len(cuts) - 1):
        faces.append((2 * k, 2 * k + 2, 2 * k + 3, 2 * k + 1))
    return vertices, faces, uv

In [ ]:
# Celda 5
# Start from the LOD3 meshes. For the textured facades: drop the thin plates (the photo shows them now)
# and give the photo to the other elements, projected straight from the front (each vertex takes the uv of its u, z).
with open(LOD3_JSON, encoding="utf-8") as f:
    meshes = json.load(f)["parts"]

by_facade = {texture["facade"]: texture for texture in TEXTURED}
x0, y0, z0 = ORIGIN
kept = []
dropped = 0
for mesh in meshes:
    texture = by_facade.get(mesh.get("facade"))
    if texture is None:
        kept.append(mesh)
        continue
    if mesh["thin"]:
        dropped += 1
        continue
    line = LineString(texture["line"])
    mesh["uv"] = []
    for x, y, z in mesh["vertices"]:
        u = line.project(shapely.Point(x + x0, y + y0))
        mesh["uv"].append(photo_uv(texture, u, z + z0))
    mesh["image"] = texture["path"]
    kept.append(mesh)
print(len(meshes), "LOD3 meshes |", dropped, "thin plates dropped |", len(kept), "kept")

In [ ]:
# Celda 6
# Add the photo planes over the walls:
for texture in TEXTURED:
    for i, (u_from, u_to, z_from, z_to) in enumerate(texture["walls"]):
        vertices, faces, uv = wall_plane(texture, u_from, u_to, z_from, z_to)
        kept.append({"name": "foto_" + os.path.basename(texture["image"])[:-4] + "_" + str(i + 1),
                     "parcel": "", "group": "conjunto", "lod": 3, "height_m": 0.0,
                     "height_method": "foto rectificada", "vertices": vertices, "faces": faces,
                     "uv": uv, "image": texture["path"]})
print(len(kept), "meshes in total")

In [ ]:
# Celda 7
# Save and run Blender:
with open(JSON_FILE, "w", encoding="utf-8") as f:
    json.dump({"origin": ORIGIN, "parts": kept}, f)

command = [BLENDER, "-b", "--factory-startup", "--python", BPY_SCRIPT, "--", JSON_FILE, BLEND_FILE, GLB_FILE]
result = subprocess.run(command, capture_output=True, text=True)
for line_text in result.stdout.splitlines():
    if "Objects" in line_text or "Error" in line_text or "Saved" in line_text:
        print(line_text)
print("Return code:", result.returncode)
if result.returncode != 0:
    print(result.stderr[-2000:])
for path in [BLEND_FILE, GLB_FILE]:
    print(path, round(os.path.getsize(path) / 1e6, 2), "MB")